In [ ]:

import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from data.get_data import get_dataframe
from data_processor.calculate_stats import calculate_statistics
from data_processor.data_categorising import categories_columns
from data_processor.data_cleaner import clean_data
from data_processor.fight_stats import finalProcessingForFighter, calculateAverages
from data_processor.data_types_fixes import check_and_process_data_type, drop_col_for_training

In [ ]:
og_df = get_dataframe('original.csv')

In [ ]:
cleaned_df = clean_data(og_df)

In [ ]:
stats_df = calculate_statistics(cleaned_df)

In [ ]:
processed_df = finalProcessingForFighter(stats_df)

In [ ]:
processed_df = check_and_process_data_type(processed_df)

In [ ]:
avg_df = calculateAverages(processed_df)

In [ ]:
cat_df = categories_columns(avg_df)

In [ ]:
df_for_training = drop_col_for_training(cat_df)

In [ ]:
df_for_training = df_for_training.sort_index()
df_for_training

In [ ]:
X = df_for_training.drop("Target", axis=1)
y = df_for_training["Target"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [21]:
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.5,
    colsample_bytree=0.9,
    early_stopping_rounds=100
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

y_preds = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]

[0]	validation_0-logloss:0.69271
[50]	validation_0-logloss:0.68676
[100]	validation_0-logloss:0.68410
[150]	validation_0-logloss:0.68314
[200]	validation_0-logloss:0.68274
[250]	validation_0-logloss:0.68274
[300]	validation_0-logloss:0.68229
[350]	validation_0-logloss:0.68250
[400]	validation_0-logloss:0.68291
[431]	validation_0-logloss:0.68307


In [ ]:
y_preds = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]

acc = metrics.accuracy_score(y_test, y_preds)
prec = metrics.precision_score(y_test, y_preds)
roc_acc = float(metrics.roc_auc_score(y_test, y_preds))

roc_acc, acc, prec

(0.5500299438817388, 0.5512415349887133, 0.5412115193644489)